# Plot a building footprint and its entrance label

In [ ]:
from os.path import join
import yaml
import geopandas as gpd
from random import randint
import leafmap
import pandas as pd
import ast
import re
from shapely.geometry import Point
from shapely import wkt

In [ ]:
def parse_entrance_geoms(entrance_geoms: str):
# Parse entrance_geoms string into shapely geometries (typically Points)
    entrance_points = []
    if isinstance(entrance_geoms, str):
        stripped = entrance_geoms.strip()
        if stripped.startswith("["):
            # Stringified list, e.g. "['POINT (x y)', 'POINT (x2 y2)']" or coordinate tuples
            try:
                items = ast.literal_eval(stripped)
            except (SyntaxError, ValueError):
                items = None
            if isinstance(items, (list, tuple)):
                for item in items:
                    if isinstance(item, str):
                        # Expected case: each item is a WKT string like 'POINT (x y)'
                        try:
                            geom = wkt.loads(item)
                            entrance_points.append(geom)
                        except Exception:
                            # Fall back to parsing 'POINT (x y)' manually if WKT loading fails
                            matches = re.findall(r"POINT\s*\(([^)]+)\)", item)
                            for match in matches:
                                x_str, y_str = match.split()
                                entrance_points.append(Point(float(x_str), float(y_str)))
                    else:
                        # Fallback for legacy formats where items are coordinate tuples
                        entrance_points.append(Point(item))
            else:
                # Handle stringified shapely points like "<POINT (x y)>" inside a list string
                matches = re.findall(r"POINT\s*\(([^)]+)\)", stripped)
                for match in matches:
                    x_str, y_str = match.split()
                    entrance_points.append(Point(float(x_str), float(y_str)))
        else:
            # Single WKT geometry string
            try:
                entrance_points = [wkt.loads(stripped)]
            except Exception:
                # As a last resort, try to extract POINT coordinates with regex
                matches = re.findall(r"POINT\s*\(([^)]+)\)", stripped)
                for match in matches:
                    x_str, y_str = match.split()
                    entrance_points.append(Point(float(x_str), float(y_str)))
    elif entrance_geoms is not None:
        entrance_points = list(entrance_geoms)
    return entrance_points
    
def plot_building_and_entrances(building_geom, entrance_geoms, metadata):
    entrance_gs = gpd.GeoDataFrame(geometry=gpd.GeoSeries(entrance_geoms, crs=metadata.crs)).to_crs("EPSG:4326")
    geometry_gs = gpd.GeoDataFrame(geometry=gpd.GeoSeries([building_geom], crs=metadata.crs)).to_crs("EPSG:4326")
    combined_gs = gpd.GeoDataFrame(geometry=gpd.GeoSeries(entrance_geoms + [building_geom], crs=metadata.crs)).to_crs("EPSG:4326")

    center = combined_gs.centroid.iloc[0].coords[0][::-1]
    m: leafmap.Map = leafmap.Map(center=center)
    m.add_gdf(geometry_gs, layer_name="Building", style={"color": "black", "fillOpacity": 0})
    m.add_gdf(entrance_gs, layer_name="Entrances", style={"color": "red", "radius": 6})
    
    padded_gs = combined_gs.buffer(0.001)  # Add a small buffer to ensure visibility
    m.zoom_to_gdf(padded_gs)
    return m

In [ ]:
# Load processed data from the path in config
config = yaml.safe_load(open('../config.yaml', 'r'))
processed_data_dir = config['processed_data']
metadata = gpd.read_file(join(processed_data_dir, 'nyc', 'metadata.gpkg'))

In [ ]:
# Randomly select a building and plot it with its entrances
random_row = metadata.iloc[randint(0, len(metadata)-1)]
m = plot_building_and_entrances(
    building_geom=random_row['geometry'],
    entrance_geoms=parse_entrance_geoms(random_row['entrance_geometries']),
    metadata=metadata
)
m